# 02 · Preparación de datos y prevención de fuga

**Capítulo 12 del informe.** Normalización sin eliminar URLs, números ni monedas; manejo de nulos y etiquetas; deduplicación y grupos de plantillas.

In [ ]:
# Ejecutar tanto si VS Code abre el notebook en notebooks/ como en la raíz.
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
from src.data.make_dataset import identify_file, load_spaphish, prepare, normalize_text
from IPython.display import display
path = identify_file()
print('Ejemplo de normalización:', normalize_text('  ¡Hola!  C$  100   https://example.org  '))
df, audit = load_spaphish(path)
display(df[['subject', 'body', 'label', 'group']].head(3))

In [ ]:
prepared, info = prepare(path)
print('CSV preparado:', prepared)
print('Mensajes procesados:', info['filas_preparadas'])
print('Grupos independientes:', info['plantillas_normalizadas_grupos'])

In [ ]:
from src.models.train_model import split_by_groups
train_idx, test_idx = split_by_groups(df)
train_groups = set(df.iloc[train_idx]['group'])
test_groups = set(df.iloc[test_idx]['group'])
assert not train_groups.intersection(test_groups), 'Existe fuga de plantillas'
print('Ejemplos train:', len(train_idx), '| test reservado:', len(test_idx))